# Importación de librerias necesarias
El primer paso será importar todas las librerias necesarias para completar el ejercicio. 

In [1]:
import pandas as pd
import numpy as np
import ast
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import gensim.downloader as api
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\XxDEs\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\XxDEs\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Preprocesamiento de los datos
Procedemos a cargar los datos, eliminamos aquellas filas que no tengan sinopsis o género principal y limpiamos el texto. 

In [2]:
df = pd.read_csv('./tmdb_5000_movies.csv')

def extract_main_genre(genre_str):
    try:
        genres = ast.literal_eval(genre_str)
        if len(genres) > 0:
            return genres[0]['name']
        return np.nan
    except:
        return np.nan

df['main_genre'] = df['genres'].apply(extract_main_genre)

df = df.dropna(subset=['overview', 'main_genre']).reset_index(drop=True)

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

df['clean_overview'] = df['overview'].apply(preprocess_text)

# Representación de las sinopsis
Utilizaremos GloVe para convertir la sinopsis en vectores numéricos.

In [3]:


print("Cargando modelo GloVe (esto puede tardar un momento)...")
glove_model = api.load('glove-wiki-gigaword-100')
embedding_dim = 100

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df['main_genre'])
num_classes = len(label_encoder.classes_)

max_words = 20000 
max_len = 100     

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['clean_overview'])
sequences = tokenizer.texts_to_sequences(df['clean_overview'])
X_padded = pad_sequences(sequences, maxlen=max_len)

word_index = tokenizer.word_index
embedding_matrix = np.zeros((max_words, embedding_dim))

for word, i in word_index.items():
    if i < max_words:
        if word in glove_model:
            embedding_matrix[i] = glove_model[word]
    

Cargando modelo GloVe (esto puede tardar un momento)...
[==================================================] 100.0% 128.1/128.1MB downloaded


# Construcción del modelo, optimización e hiperparámetros
Definimos una arquitectura con una capa LSTM, entrenamos el modelo, optimizamos y ajustamos hiperparámetros. 

In [4]:
X_train, X_val, y_train, y_val = train_test_split(X_padded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, 
              weights=[embedding_matrix], input_length=max_len, trainable=False),
    
    LSTM(128, return_sequences=False),
    
    Dense(64, activation='relu'),
    
    Dropout(0.5),
    
    Dense(num_classes, activation='softmax')
])

learning_rate = 0.001
batch_size = 64
epochs = 20

optimizer = Adam(learning_rate=learning_rate)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()

history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, y_val),
    class_weight=class_weight_dict 
)

c:\Users\XxDEs\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,000,000 (7.63 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,000,000 (7.63 MB)

Epoch 1/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 54ms/step - accuracy: 0.0558 - loss: 3.0175 - val_accuracy: 0.1581 - val_loss: 2.9239
Epoch 2/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.0893 - loss: 2.9094 - val_accuracy: 0.0880 - val_loss: 2.8564
Epoch 3/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.1019 - loss: 2.7445 - val_accuracy: 0.1319 - val_loss: 2.7343
Epoch 4/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - accuracy: 0.0896 - loss: 2.6018 - val_accuracy: 0.0586 - val_loss: 2.7772
Epoch 5/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step - accuracy: 0.0794 - loss: 2.4074 - val_accuracy: 0.0848 - val_loss: 2.7075
Epoch 6/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.1100 - loss: 2.1727 - val_accuracy: 0.0880 - val_loss: 2.5875
Epoch 7/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.1158 - loss: 2.1069 - val_accuracy: 0.0963 - val_loss: 2.6005
Epoch 8/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.1263 - loss: 1.9919 - val_accuracy: 0.1141 - v

# Evaluación del modelo
Como último paso evaluaremos el modelo. 

In [5]:

y_pred_probs = model.predict(X_val)
y_pred = np.argmax(y_pred_probs, axis=1)

print("Reporte de Clasificación del Modelo Optimizado (LSTM + GloVe + Class Weights):")
print(classification_report(y_val, y_pred, target_names=label_encoder.classes_, zero_division=0))


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step
Reporte de Clasificación del Modelo Optimizado (LSTM + GloVe + Class Weights):
                 precision    recall  f1-score   support

         Action       0.54      0.22      0.31       151
      Adventure       0.22      0.21      0.21        68
      Animation       0.20      0.44      0.28        25
         Comedy       0.47      0.22      0.30       209
          Crime       0.17      0.41      0.24        39
    Documentary       0.19      0.47      0.27        17
          Drama       0.56      0.15      0.23       241
         Family       0.12      0.18      0.14        11
        Fantasy       0.11      0.12      0.12        24
        Foreign       0.00      0.00      0.00         0
        History       0.00      0.00      0.00         5
         Horror       0.26      0.47      0.33        60
          Music       0.07      0.29      0.11         7
        Mystery       0.00      0.00      0.00         8
        Romance       0.0